In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import tqdm
from torch.utils.tensorboard import SummaryWriter

c:\Source\TEAM-PJ-DEEP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
df_train = pd.read_csv(('../data/train_multilabel.csv'))
df_val = pd.read_csv(('../data/test_multilabel.csv'))
df_test = pd.read_csv(('../data/valid_multilabel.csv'))


In [13]:
label_cols = df_train.columns[6:].tolist()

In [14]:
#데이터 셋 만들기(멀티라벨 용으로)

class ComplainDataset(Dataset):
    def __init__(self, df, tokenizer, label_cols, max_length=128):
        self.df = df
        self.tokenizer = tokenizer
        self.label_cols = label_cols
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = self.df.loc[idx,'text']
        
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        label = self.df.loc[idx, self.label_cols].values.astype("float32")

        item = {
            "input_ids": encoding['input_ids'].squeeze(0),
            "attention_mask": encoding['attention_mask'].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.float32) 
        }
        return item

In [15]:
# 한국어 전용 토크나이져
tokenizer = AutoTokenizer.from_pretrained("klue/roberta-base")

In [16]:
#dataset 만들기
train_dataset = ComplainDataset(df_train, tokenizer, label_cols)
val_dataset = ComplainDataset(df_val, tokenizer, label_cols)
test_dataset = ComplainDataset(df_test, tokenizer, label_cols)

In [17]:
#data_loaders 만들기
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

In [18]:
# 모델 생성
model = AutoModelForSequenceClassification.from_pretrained(
    "klue/roberta-base",
    num_labels=len(label_cols),
    problem_type="multi_label_classification"
)

for params in model.parameters():
    params.requires_grad = False

# for param in model.roberta.encoder.layer[11].parameters():
#     param.requires_grad = True

for param in model.classifier.parameters():
    param.requires_grad = True
    
for name,module in model.named_parameters():
    print(name, module.requires_grad)

model

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1109.89it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: klue/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. C

roberta.embeddings.word_embeddings.weight False
roberta.embeddings.token_type_embeddings.weight False
roberta.embeddings.LayerNorm.weight False
roberta.embeddings.LayerNorm.bias False
roberta.embeddings.position_embeddings.weight False
roberta.encoder.layer.0.attention.self.query.weight False
roberta.encoder.layer.0.attention.self.query.bias False
roberta.encoder.layer.0.attention.self.key.weight False
roberta.encoder.layer.0.attention.self.key.bias False
roberta.encoder.layer.0.attention.self.value.weight False
roberta.encoder.layer.0.attention.self.value.bias False
roberta.encoder.layer.0.attention.output.dense.weight False
roberta.encoder.layer.0.attention.output.dense.bias False
roberta.encoder.layer.0.attention.output.LayerNorm.weight False
roberta.encoder.layer.0.attention.output.LayerNorm.bias False
roberta.encoder.layer.0.intermediate.dense.weight False
roberta.encoder.layer.0.intermediate.dense.bias False
roberta.encoder.layer.0.output.dense.weight False
roberta.encoder.layer.

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(32000, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [19]:
writer = SummaryWriter()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.BCEWithLogitsLoss()

model.to(device)

best_val_loss = 100000000
epochs = 60
early_stop_epochs = 3
early_stop_counter = 0
count = 0

for epoch in range(epochs):
    train_tqdm = tqdm.tqdm(train_loader)
    model.train()
    train_loss_sum = 0

    for batch in train_tqdm:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        train_loss_sum += loss.item()

        writer.add_scalar("Loss/train_step", loss.item(), count)
        count += 1

        train_tqdm.set_postfix(loss=f"{loss.item():.4f}")

    avg_train_loss = train_loss_sum / len(train_loader)
    print("avg_train_loss",avg_train_loss)

    model.eval()    
    all_preds = []
    all_labels = []
    val_loss_sum = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits

            loss = criterion(logits, labels)
            val_loss_sum += loss.item()    

            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).float()

            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu())

        avg_val_loss = val_loss_sum / len(val_loader)
    print( "avg_val_loss",avg_val_loss)
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "multiLabel_best_model.pt")
        early_stop_counter = 0
    else:
        early_stop_counter += 1

        if early_stop_counter >= early_stop_epochs:
            print("Early stopping triggered.")
            break

    

    



100%|██████████| 505/505 [00:16<00:00, 29.84it/s, loss=0.1810]


avg_train_loss 0.38159155370575365
avg_val_loss 0.14402805296879895


100%|██████████| 505/505 [00:17<00:00, 29.14it/s, loss=0.1349]


avg_train_loss 0.14384840620626319
avg_val_loss 0.09656547872150469


100%|██████████| 505/505 [00:17<00:00, 28.78it/s, loss=0.1290]


...

avg_train_loss 0.09122715745822038
avg_val_loss 0.07070972800910848


100%|██████████| 505/505 [00:17<00:00, 29.50it/s, loss=0.0921]


avg_train_loss 0.09074868567214153
avg_val_loss 0.07040578297942689


100%|██████████| 505/505 [00:17<00:00, 29.55it/s, loss=0.0867]


avg_train_loss 0.08976391000617849
avg_val_loss 0.06978123693627382


In [20]:
model.eval()

test_loss_sum = 0
all_preds = []
all_labels = []
test_tqdm = tqdm.tqdm(test_loader, desc="Test")
with torch.no_grad():
    for batch in test_tqdm:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        loss = criterion(logits, labels)
        test_loss_sum += loss.item()
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).float()
        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())
        test_tqdm.set_postfix(loss=f"{loss.item():.4f}")
avg_test_loss = test_loss_sum / len(test_loader)

avg_test_loss

Test: 100%|██████████| 127/127 [00:04<00:00, 29.47it/s, loss=0.0917]


0.08748483804501886

In [24]:
text = "전입신고 하고싶은데"

model.eval()

encoding = tokenizer(
    text,
    truncation=True,
    padding="max_length",
    max_length=128,
    return_tensors="pt"
)
input_ids = encoding["input_ids"].to(device)
attention_mask = encoding["attention_mask"].to(device)
with torch.no_grad():
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits
    probs = torch.sigmoid(logits).squeeze(0).cpu().numpy()
pred_labels = []
for col, prob in zip(label_cols, probs):
    if prob >= 0.2:
        pred_labels.append((col, float(prob)))
pred_labels = sorted(pred_labels, key=lambda x: x[1], reverse=True)

print(pred_labels)
print(probs)

[('문서:신분증', 0.6542903780937195), ('기관:읍·면·동 행정복지센터', 0.2828863859176636), ('기관:시·군·구청/읍·면사무소', 0.2826533913612366), ('부서:민원여권과/가족관계등록팀', 0.27247852087020874)]
[0.02384028 0.01823868 0.03359246 0.02934508 0.0285514  0.0178285
 0.01882259 0.01277514 0.10468025 0.2826534  0.04304761 0.2828864
 0.0457866  0.04151337 0.04041459 0.02289516 0.02804018 0.0240784
 0.01485724 0.02736523 0.00707627 0.01737101 0.02200457 0.0456906
 0.04244431 0.02684668 0.01239803 0.02071552 0.05322786 0.05387755
 0.01538246 0.00964854 0.02354656 0.00656921 0.00799378 0.01039962
 0.0348682  0.01710804 0.00958229 0.02178866 0.00666005 0.01089837
 0.01830233 0.01586682 0.01412666 0.01964162 0.01015082 0.00981043
 0.02030421 0.04294971 0.05347318 0.0524761  0.03922434 0.00971042
 0.04380579 0.04317722 0.0225038  0.01326219 0.00905643 0.01220646
 0.01880965 0.01462161 0.01362584 0.09803453 0.0298361  0.00640468
 0.0417277  0.6542904  0.02849273 0.03575309 0.02395882 0.0187469
 0.01526568 0.0188719  0.04827068 0.026339